# Week 7 Day 1 — Embeddings

## Objective

Understand what embedding models produce and how semantic similarity
emerges from embedding vectors.

In [1]:
from openai import OpenAI

client = OpenAI()

text = "The dog is a friendly pet."

response = client.embeddings.create(
    model="text-embedding-3-small",
    input=text
)

embedding = response.data[0].embedding

print("Text:", text)
print("Embedding dimensions:", len(embedding))
print("First 10 values:", embedding[:10])

Text: The dog is a friendly pet.
Embedding dimensions: 1536
First 10 values: [0.0108642578125, -0.038330078125, -0.03155517578125, 0.02349853515625, 0.021636962890625, 0.00464630126953125, -0.0233154296875, 0.04742431640625, -0.01788330078125, -0.016510009765625]


## Activity 2 — Compare Embedding Models

We will embed the same sentence using:

1. text-embedding-3-small
2. text-embedding-3-large

We will compare their vector dimensions and values.

In [2]:
text = "The dog is a friendly pet."

small_response = client.embeddings.create(
    model="text-embedding-3-small",
    input=text
)

large_response = client.embeddings.create(
    model="text-embedding-3-large",
    input=text
)

small_embedding = small_response.data[0].embedding
large_embedding = large_response.data[0].embedding

print("Text:", text)
print()
print("SMALL MODEL")
print("Dimensions:", len(small_embedding))
print("First 5 values:", small_embedding[:5])
print()
print("LARGE MODEL")
print("Dimensions:", len(large_embedding))
print("First 5 values:", large_embedding[:5])

Text: The dog is a friendly pet.

SMALL MODEL
Dimensions: 1536
First 5 values: [0.0108795166015625, -0.03826904296875, -0.03155517578125, 0.0234832763671875, 0.021636962890625]

LARGE MODEL
Dimensions: 3072
First 5 values: [-0.02386474609375, 0.039306640625, -0.00763702392578125, 0.006237030029296875, 0.015411376953125]


## Activity 3 — Semantic Discrimination

Compare cosine similarity for:

- Same meaning
- Related meaning
- Distant meaning

We will run the experiment with both embedding models.

In [3]:
import numpy as np

def cosine_similarity(a, b):
    a = np.array(a)
    b = np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


texts = {
    "original": "The dog is a friendly pet.",
    "same": "Dogs are friendly household pets.",
    "related": "Cats are common domestic animals.",
    "distant": "The stock market closed higher today."
}


def get_embedding(text, model):
    response = client.embeddings.create(
        model=model,
        input=text
    )
    return response.data[0].embedding


for model in ["text-embedding-3-small", "text-embedding-3-large"]:
    print("=" * 60)
    print(model)

    original = get_embedding(texts["original"], model)

    for name in ["same", "related", "distant"]:
        other = get_embedding(texts[name], model)
        score = cosine_similarity(original, other)

        print(f"{name:10s}: {score:.4f}")

text-embedding-3-small


same      : 0.7296
related   : 0.3540
distant   : 0.1201
text-embedding-3-large
same      : 0.7657
related   : 0.3690
distant   : 0.1143


## Activity 4 — Embedding Dimension and Storage

We compare the raw vector storage required by
1536-dimensional and 3072-dimensional embeddings.

In [4]:
dimensions = {
    "text-embedding-3-small": 1536,
    "text-embedding-3-large": 3072,
}

bytes_per_dimension = 4  # float32

document_counts = [1_000, 10_000, 1_000_000, 10_000_000]

for model, dims in dimensions.items():
    print(f"\n{model}")
    print(f"Dimensions: {dims}")

    for count in document_counts:
        storage_bytes = count * dims * bytes_per_dimension
        storage_mb = storage_bytes / (1024 ** 2)
        storage_gb = storage_bytes / (1024 ** 3)

        print(
            f"{count:>10,} vectors -> "
            f"{storage_mb:,.2f} MB "
            f"({storage_gb:,.2f} GB)"
        )


text-embedding-3-small
Dimensions: 1536
     1,000 vectors -> 5.86 MB (0.01 GB)
    10,000 vectors -> 58.59 MB (0.06 GB)
 1,000,000 vectors -> 5,859.38 MB (5.72 GB)
10,000,000 vectors -> 58,593.75 MB (57.22 GB)

text-embedding-3-large
Dimensions: 3072
     1,000 vectors -> 11.72 MB (0.01 GB)
    10,000 vectors -> 117.19 MB (0.11 GB)
 1,000,000 vectors -> 11,718.75 MB (11.44 GB)
10,000,000 vectors -> 117,187.50 MB (114.44 GB)


## Activity 5 — Semantic Geometry

We will embed a small corpus of 10 animals and examine
their semantic relationships using cosine similarity.

The corpus contains:
- 2 mammals
- 2 birds
- 2 fish
- 2 reptiles
- 2 insects

In [8]:
animals = [
    {
        "id": "dog",
        "category": "mammal",
        "text": "Dogs are domesticated mammals commonly kept as pets and are known for their loyalty and companionship."
    },
    {
        "id": "cat",
        "category": "mammal",
        "text": "Cats are domesticated mammals often kept as pets and are known for their independence and hunting instincts."
    },
    {
        "id": "eagle",
        "category": "bird",
        "text": "Eagles are large birds of prey known for their excellent vision, powerful wings, and ability to hunt."
    },
    {
        "id": "penguin",
        "category": "bird",
        "text": "Penguins are flightless birds that live mainly in the Southern Hemisphere and are excellent swimmers."
    },
    {
        "id": "salmon",
        "category": "fish",
        "text": "Salmon are fish that live in oceans and freshwater and are known for swimming upstream to spawn."
    },
    {
        "id": "shark",
        "category": "fish",
        "text": "Sharks are predatory fish that live in oceans and use powerful senses to locate prey."
    },
    {
        "id": "snake",
        "category": "reptile",
        "text": "Snakes are reptiles with elongated bodies that move without legs and include many different species."
    },
    {
        "id": "gecko",
        "category": "reptile",
        "text": "Geckos are small reptiles that are often kept as pets and are known for their ability to climb surfaces."
    },
    {
        "id": "ant",
        "category": "insect",
        "text": "Ants are social insects that live in colonies and work together to gather food and maintain their nests."
    },
    {
        "id": "bee",
        "category": "insect",
        "text": "Bees are social insects that collect nectar and pollen from flowers and play an important role in pollination."
    }
]

print("Number of animals:", len(animals))

for animal in animals:
    print(
        animal["id"],
        "->",
        animal["category"]
    )

Number of animals: 10
dog -> mammal
cat -> mammal
eagle -> bird
penguin -> bird
salmon -> fish
shark -> fish
snake -> reptile
gecko -> reptile
ant -> insect
bee -> insect


### Generate embeddings for the animal corpus

We use `text-embedding-3-small` to convert each animal description
into a 1536-dimensional vector.

In [6]:
def embed_text(text, model="text-embedding-3-small"):
    response = client.embeddings.create(
        model=model,
        input=text
    )
    return response.data[0].embedding

In [7]:
animal_embeddings = []

for animal in animals:
    vector = embed_text(animal["text"])

    animal_embeddings.append({
        "id": animal["id"],
        "category": animal["category"],
        "text": animal["text"],
        "vector": vector
    })

print("Animals embedded:", len(animal_embeddings))
print("Vector dimensions:", len(animal_embeddings[0]["vector"]))

Animals embedded: 10
Vector dimensions: 1536


Step 2C — Inspect one complete record

In [8]:
first = animal_embeddings[0]

print("ID:", first["id"])
print("Category:", first["category"])
print("Text:", first["text"])
print("Vector dimensions:", len(first["vector"]))
print("First 5 vector values:", first["vector"][:5])

ID: dog
Category: mammal
Text: Dogs are domesticated mammals commonly kept as pets and are known for their loyalty and companionship.
Vector dimensions: 1536
First 5 vector values: [0.038818359375, -0.016876220703125, 0.0037689208984375, 0.01148223876953125, -0.00914764404296875]


Step 3A — Calculate the matrix

## Activity 5 — Pairwise Cosine Similarity

Calculate cosine similarity between every pair of animal embeddings
to observe the semantic geometry of the embedding space.

In [9]:
import numpy as np

def cosine_similarity(a, b):
    a = np.array(a)
    b = np.array(b)

    return np.dot(a, b) / (
        np.linalg.norm(a) * np.linalg.norm(b)
    )


animal_ids = [animal["id"] for animal in animal_embeddings]

vectors = np.array([
    animal["vector"]
    for animal in animal_embeddings
])

similarity_matrix = np.zeros(
    (len(vectors), len(vectors))
)

for i in range(len(vectors)):
    for j in range(len(vectors)):
        similarity_matrix[i, j] = cosine_similarity(
            vectors[i],
            vectors[j]
        )

print("Matrix shape:", similarity_matrix.shape)
print("Diagonal:", np.diag(similarity_matrix))

Matrix shape: (10, 10)
Diagonal: [1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


Step 3B — Display the matrix

In [10]:
import pandas as pd

similarity_df = pd.DataFrame(
    similarity_matrix,
    index=animal_ids,
    columns=animal_ids
)

similarity_df.round(3)

,dog,cat,eagle,penguin,salmon,shark,snake,gecko,ant,bee
dog,1.000,0.626,0.235,0.176,0.147,0.195,0.172,0.350,0.240,0.191
cat,0.626,1.000,0.348,0.218,0.191,0.281,0.205,0.415,0.246,0.190
eagle,0.235,0.348,1.000,0.351,0.173,0.398,0.227,0.308,0.256,0.291
penguin,0.176,0.218,0.351,1.000,0.303,0.308,0.194,0.256,0.228,0.221
salmon,0.147,0.191,0.173,0.303,1.000,0.374,0.239,0.133,0.205,0.148
shark,0.195,0.281,0.398,0.308,0.374,1.000,0.235,0.152,0.243,0.208
snake,0.172,0.205,0.227,0.194,0.239,0.235,1.000,0.335,0.197,0.184
gecko,0.350,0.415,0.308,0.256,0.133,0.152,0.335,1.000,0.187,0.150
ant,0.240,0.246,0.256,0.228,0.205,0.243,0.197,0.187,1.000,0.533
bee,0.191,0.190,0.291,0.221,0.148,0.208,0.184,0.150,0.533,1.000


Activity 5 — Step 4: Find the Top-3 Semantic Neighbours

In [11]:
top_k = 3

for i, animal_id in enumerate(animal_ids):
    similarities = similarity_matrix[i].copy()

    # Exclude the animal itself
    similarities[i] = -1

    top_indices = np.argsort(similarities)[::-1][:top_k]

    print(f"\n{animal_id}:")
    
    for idx in top_indices:
        print(
            f"  {animal_ids[idx]:10s} "
            f"similarity={similarities[idx]:.3f}"
        )


dog:
  cat        similarity=0.626
  gecko      similarity=0.350
  ant        similarity=0.240

cat:
  dog        similarity=0.626
  gecko      similarity=0.415
  eagle      similarity=0.348

eagle:
  shark      similarity=0.398
  penguin    similarity=0.351
  cat        similarity=0.348

penguin:
  eagle      similarity=0.351
  shark      similarity=0.308
  salmon     similarity=0.303

salmon:
  shark      similarity=0.374
  penguin    similarity=0.303
  snake      similarity=0.239

shark:
  eagle      similarity=0.398
  salmon     similarity=0.374
  penguin    similarity=0.308

snake:
  gecko      similarity=0.335
  salmon     similarity=0.239
  shark      similarity=0.235

gecko:
  cat        similarity=0.415
  dog        similarity=0.350
  snake      similarity=0.335

ant:
  bee        similarity=0.533
  eagle      similarity=0.256
  cat        similarity=0.246

bee:
  ant        similarity=0.533
  eagle      similarity=0.291
  penguin    similarity=0.221


6. Now let's calculate same-category retrieval

In [12]:
category_by_id = {
    animal["id"]: animal["category"]
    for animal in animals
}

total_same_category = 0
total_neighbours = 0

for i, animal_id in enumerate(animal_ids):

    similarities = similarity_matrix[i].copy()
    similarities[i] = -1

    top_indices = np.argsort(similarities)[::-1][:3]

    same_category = 0

    for idx in top_indices:
        neighbour_id = animal_ids[idx]

        if category_by_id[animal_id] == category_by_id[neighbour_id]:
            same_category += 1

    total_same_category += same_category
    total_neighbours += 3

    print(
        f"{animal_id:10s} "
        f"same-category neighbours: "
        f"{same_category}/3"
    )

fraction = total_same_category / total_neighbours

print("\nOverall same-category fraction:", round(fraction, 3))
print("Percentage:", round(fraction * 100, 1), "%")

dog        same-category neighbours: 1/3
cat        same-category neighbours: 1/3
eagle      same-category neighbours: 1/3
penguin    same-category neighbours: 1/3
salmon     same-category neighbours: 1/3
shark      same-category neighbours: 1/3
snake      same-category neighbours: 1/3
gecko      same-category neighbours: 1/3
ant        same-category neighbours: 1/3
bee        same-category neighbours: 1/3

Overall same-category fraction: 0.333
Percentage: 33.3 %


Activity 6 — Pet Query Retrieval

In [13]:
query = "What animals are commonly kept as pets?"

query_vector = embed_text(query)

query_scores = []

for animal in animal_embeddings:
    score = cosine_similarity(
        query_vector,
        animal["vector"]
    )

    query_scores.append({
        "id": animal["id"],
        "category": animal["category"],
        "score": score,
        "text": animal["text"]
    })

query_scores = sorted(
    query_scores,
    key=lambda x: x["score"],
    reverse=True
)

print("Query:", query)
print("\nTop 5 retrieved animals:")

for rank, result in enumerate(query_scores[:5], start=1):
    print(
        f"{rank}. {result['id']:10s} "
        f"category={result['category']:8s} "
        f"score={result['score']:.4f}"
    )

Query: What animals are commonly kept as pets?

Top 5 retrieved animals:
1. dog        category=mammal   score=0.5769
2. cat        category=mammal   score=0.4980
3. gecko      category=reptile  score=0.4121
4. snake      category=reptile  score=0.2480
5. penguin    category=bird     score=0.2292


In [14]:
relevant = {"dog", "cat", "gecko"}

top_3 = [
    result["id"]
    for result in query_scores[:3]
]

relevant_retrieved = [
    animal_id
    for animal_id in top_3
    if animal_id in relevant
]

precision_at_3 = len(relevant_retrieved) / 3

print("Top-3 retrieved:", top_3)
print("Relevant retrieved:", relevant_retrieved)
print("Precision@3:", round(precision_at_3, 3))
print("Precision@3 (%):", round(precision_at_3 * 100, 1))

Top-3 retrieved: ['dog', 'cat', 'gecko']
Relevant retrieved: ['dog', 'cat', 'gecko']
Precision@3: 1.0
Precision@3 (%): 100.0


In [15]:
total_relevant = len(relevant)

recall_at_3 = len(relevant_retrieved) / total_relevant

print("Total relevant documents:", total_relevant)
print("Relevant documents retrieved:", len(relevant_retrieved))
print("Recall@3:", round(recall_at_3, 3))
print("Recall@3 (%):", round(recall_at_3 * 100, 1))

Total relevant documents: 3
Relevant documents retrieved: 3
Recall@3: 1.0
Recall@3 (%): 100.0


4. Before connecting: check your Qdrant environment

In [16]:
import os

print("QDRANT_URL set:", bool(os.getenv("QDRANT_URL")))
print("QDRANT_API_KEY set:", bool(os.getenv("QDRANT_API_KEY")))

QDRANT_URL set: False
QDRANT_API_KEY set: False


In [17]:
import qdrant_client
import importlib.metadata

print("qdrant-client imported successfully")
print("Version:", importlib.metadata.version("qdrant-client"))

qdrant-client imported successfully
Version: 1.19.1


In [18]:
import os
from getpass import getpass

qdrant_url = input("Enter Qdrant URL: ").strip()
qdrant_api_key = getpass("Enter Qdrant API key: ")

os.environ["QDRANT_URL"] = qdrant_url
os.environ["QDRANT_API_KEY"] = qdrant_api_key

print("QDRANT_URL set:", bool(os.getenv("QDRANT_URL")))
print("QDRANT_API_KEY set:", bool(os.getenv("QDRANT_API_KEY")))

QDRANT_URL set: True
QDRANT_API_KEY set: True


In [19]:
import os
from qdrant_client import QdrantClient

qdrant = QdrantClient(
    url=os.environ["QDRANT_URL"],
    api_key=os.environ["QDRANT_API_KEY"]
)

print("Qdrant client created successfully")

Qdrant client created successfully


/voc/work/.local/lib/python3.10/site-packages/qdrant_client/qdrant_remote.py:277: UserWarning: Failed to obtain server version. Unable to check client-server compatibility. Set check_compatibility=False to skip version check.
  show_warning(


In [20]:
collections = qdrant.get_collections()

print("Connected to Qdrant successfully")
print("Existing collections:")

for collection in collections.collections:
    print("-", collection.name)

UnexpectedResponse: Unexpected Response: 403 (Forbidden)
Raw response content:
b'{"error":"forbidden"}'

In [ ]:
import os

url = os.getenv("QDRANT_URL", "")

print("URL configured:", bool(url))
print("Starts with HTTPS:", url.startswith("https://"))
print("Contains qdrant.io:", "qdrant.io" in url)
print("Ends with slash:", url.endswith("/"))

URL configured: True
Starts with HTTPS: True
Contains qdrant.io: True
Ends with slash: False


In [ ]:
api_key = os.getenv("QDRANT_API_KEY", "")

print("API key configured:", bool(api_key))
print("API key length:", len(api_key))

API key configured: True
API key length: 179


In [ ]:
from qdrant_client import QdrantClient

qdrant = QdrantClient(
    url=os.environ["QDRANT_URL"],
    api_key=os.environ["QDRANT_API_KEY"],
)

print("Client created")

Client created


In [21]:
collections = qdrant.get_collections()

print("Connection successful")
print("Number of collections:", len(collections.collections))

UnexpectedResponse: Unexpected Response: 403 (Forbidden)
Raw response content:
b'{"error":"forbidden"}'

In [ ]:
import os

url = os.getenv("QDRANT_URL", "")

print("URL configured:", bool(url))
print("Starts with HTTPS:", url.startswith("https://"))
print("Contains qdrant.io:", "qdrant.io" in url)
print("Ends with slash:", url.endswith("/"))

URL configured: True
Starts with HTTPS: True
Contains qdrant.io: True
Ends with slash: False


In [22]:
import os

url = os.getenv("QDRANT_URL", "")

print("URL configured:", bool(url))
print("Starts with HTTPS:", url.startswith("https://"))
print("Contains qdrant.io:", "qdrant.io" in url)
print("Ends with slash:", url.endswith("/"))


URL configured: True
Starts with HTTPS: True
Contains qdrant.io: True
Ends with slash: False


In [23]:
api_key = os.getenv("QDRANT_API_KEY", "")

print("API key configured:", bool(api_key))
print("API key length:", len(api_key))


API key configured: True
API key length: 179


In [24]:
from qdrant_client import QdrantClient
import os

qdrant = QdrantClient(
    url=os.environ["QDRANT_URL"],
    api_key=os.environ["QDRANT_API_KEY"]
)

print("Qdrant client created")

collections = qdrant.get_collections()

print("Qdrant connection successful")
print(collections)

Qdrant client created


UnexpectedResponse: Unexpected Response: 403 (Forbidden)
Raw response content:
b'{"error":"forbidden"}'

In [25]:
import os

url = os.getenv("QDRANT_URL", "")
api_key = os.getenv("QDRANT_API_KEY", "")

print("URL configured:", bool(url))
print("API key configured:", bool(api_key))
print("API key length:", len(api_key))

URL configured: True
API key configured: True
API key length: 179


In [26]:
from qdrant_client import QdrantClient
import os

qdrant = QdrantClient(
    url=os.environ["QDRANT_URL"],
    api_key=os.environ["QDRANT_API_KEY"]
)

print("Qdrant client created")

collections = qdrant.get_collections()

print("Qdrant connection successful")
print(collections)

Qdrant client created


UnexpectedResponse: Unexpected Response: 403 (Forbidden)
Raw response content:
b'{"error":"forbidden"}'

In [1]:
import os

print("Python version:", __import__("sys").version)
print("QDRANT_URL configured:", bool(os.getenv("QDRANT_URL")))
print("QDRANT_API_KEY configured:", bool(os.getenv("QDRANT_API_KEY")))

Python version: 3.10.14 (main, Apr  6 2024, 18:45:05) [GCC 9.4.0]
QDRANT_URL configured: False
QDRANT_API_KEY configured: False


In [2]:
import os
from getpass import getpass

os.environ["QDRANT_URL"] = input("Enter QDRANT_URL: ").strip()
os.environ["QDRANT_API_KEY"] = getpass("Enter QDRANT_API_KEY: ")

print("QDRANT_URL configured:", bool(os.environ["QDRANT_URL"]))
print("QDRANT_API_KEY configured:", bool(os.environ["QDRANT_API_KEY"]))
print("API key length:", len(os.environ["QDRANT_API_KEY"]))

QDRANT_URL configured: True
QDRANT_API_KEY configured: True
API key length: 176


Step 3 — Create the Qdrant client

In [3]:
from qdrant_client import QdrantClient
import os

qdrant = QdrantClient(
    url=os.environ["QDRANT_URL"],
    api_key=os.environ["QDRANT_API_KEY"]
)

print("Qdrant client created")

Qdrant client created


Qdrant client created

In [4]:
collections = qdrant.get_collections()

print("Qdrant connection successful")
print(collections)

Qdrant connection successful
collections=[]


Step 1 — Create the collection

In [5]:
from qdrant_client.models import Distance, VectorParams

collection_name = "w7_animals_gorthi"

qdrant.create_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(
        size=1536,
        distance=Distance.COSINE,
    ),
)

print("Collection created:", collection_name)

Collection created: w7_animals_gorthi


Step 2 — Verify the collection configuration

In [6]:
info = qdrant.get_collection(collection_name)

print("Collection:", collection_name)
print("Vector size:", info.config.params.vectors.size)
print("Distance:", info.config.params.vectors.distance)
print("Points count:", info.points_count)

Collection: w7_animals_gorthi
Vector size: 1536
Distance: Cosine
Points count: 0


Before we upload the 10 animals

There is one important thing: when we restarted the notebook kernel, Python variables from the earlier cells were cleared.

Our 10 embeddings were previously stored in the notebook variable:

In [9]:
print("Animals:", len(animals))
print("IDs:", [a["id"] for a in animals])

Animals: 10
IDs: ['dog', 'cat', 'eagle', 'penguin', 'salmon', 'shark', 'snake', 'gecko', 'ant', 'bee']


In [10]:
print("animal_embeddings exists:", "animal_embeddings" in globals())

if "animal_embeddings" in globals():
    print("Number of embeddings:", len(animal_embeddings))
    print("Vector dimensions:", len(animal_embeddings[0]["vector"]))

animal_embeddings exists: False


In [11]:
from openai import OpenAI

client = OpenAI()

def embed_text(text, model="text-embedding-3-small"):
    response = client.embeddings.create(
        model=model,
        input=text
    )
    return response.data[0].embedding

print("OpenAI embedding client ready")

OpenAI embedding client ready


In [12]:
animal_embeddings = []

for animal in animals:
    vector = embed_text(animal["text"])

    animal_embeddings.append({
        "id": animal["id"],
        "category": animal["category"],
        "text": animal["text"],
        "vector": vector,
    })

print("Animals embedded:", len(animal_embeddings))
print("Vector dimensions:", len(animal_embeddings[0]["vector"]))

Animals embedded: 10
Vector dimensions: 1536


In [13]:
dimensions = [len(item["vector"]) for item in animal_embeddings]

print("Number of vectors:", len(dimensions))
print("Unique dimensions:", set(dimensions))
print("All vectors are 1536D:", all(d == 1536 for d in dimensions))

Number of vectors: 10
Unique dimensions: {1536}
All vectors are 1536D: True


Step 5 — Upsert the 10 animals into Qdrant
A useful Qdrant concept here is:

Point ID → unique identifier used internally by Qdrant
Vector → the 1536-dimensional embedding
Payload → our metadata (animal_id, category, text)

In [14]:
from qdrant_client.models import PointStruct

points = [
    PointStruct(
        id=i,
        vector=item["vector"],
        payload={
            "animal_id": item["id"],
            "category": item["category"],
            "text": item["text"],
        },
    )
    for i, item in enumerate(animal_embeddings)
]

qdrant.upsert(
    collection_name=collection_name,
    points=points,
)

print("Points uploaded:", len(points))

Points uploaded: 10


Step 6 — Verify the points

In [15]:
info = qdrant.get_collection(collection_name)

print("Collection:", collection_name)
print("Points count:", info.points_count)

Collection: w7_animals_gorthi
Points count: 10


Step 7 — First Qdrant retrieval

1. Create the query embedding

In [16]:
query_text = "What animals are commonly kept as pets?"

query_vector = embed_text(query_text)

print("Query:", query_text)
print("Query vector dimensions:", len(query_vector))

Query: What animals are commonly kept as pets?
Query vector dimensions: 1536


In [ ]:
2. Query Qdrant

In [17]:
results = qdrant.query_points(
    collection_name=collection_name,
    query=query_vector,
    limit=5,
    with_payload=True,
)

for rank, point in enumerate(results.points, start=1):
    print(
        f"{rank}. "
        f"{point.payload['animal_id']} | "
        f"category={point.payload['category']} | "
        f"score={point.score:.4f}"
    )

1. dog | category=mammal | score=0.5769
2. cat | category=mammal | score=0.4980
3. gecko | category=reptile | score=0.4121
4. snake | category=reptile | score=0.2480
5. penguin | category=bird | score=0.2292


In [18]:
query_text = "Which animals are good swimmers?"

query_vector = embed_text(query_text)

results = qdrant.query_points(
    collection_name=collection_name,
    query=query_vector,
    limit=5,
    with_payload=True,
)

print("Query:", query_text)

for rank, point in enumerate(results.points, start=1):
    print(
        f"{rank}. "
        f"{point.payload['animal_id']} | "
        f"category={point.payload['category']} | "
        f"score={point.score:.4f}"
    )

Query: Which animals are good swimmers?
1. penguin | category=bird | score=0.4873
2. salmon | category=fish | score=0.3632
3. shark | category=fish | score=0.3205
4. dog | category=mammal | score=0.3028
5. cat | category=mammal | score=0.2792


In [19]:
query_text = "Which animals are predators?"

query_vector = embed_text(query_text)

results = qdrant.query_points(
    collection_name=collection_name,
    query=query_vector,
    limit=5,
    with_payload=True,
)

print("Query:", query_text)

for rank, point in enumerate(results.points, start=1):
    print(
        f"{rank}. "
        f"{point.payload['animal_id']} | "
        f"category={point.payload['category']} | "
        f"score={point.score:.4f}"
    )

Query: Which animals are predators?
1. shark | category=fish | score=0.4774
2. cat | category=mammal | score=0.4350
3. eagle | category=bird | score=0.4297
4. dog | category=mammal | score=0.3385
5. snake | category=reptile | score=0.2927


In [20]:
query_text = "Which animals are social and live in groups?"

query_vector = embed_text(query_text)

results = qdrant.query_points(
    collection_name=collection_name,
    query=query_vector,
    limit=5,
    with_payload=True,
)

print("Query:", query_text)

for rank, point in enumerate(results.points, start=1):
    print(
        f"{rank}. "
        f"{point.payload['animal_id']} | "
        f"category={point.payload['category']} | "
        f"score={point.score:.4f}"
    )

Query: Which animals are social and live in groups?
1. ant | category=insect | score=0.4719
2. dog | category=mammal | score=0.4108
3. cat | category=mammal | score=0.3776
4. penguin | category=bird | score=0.3019
5. shark | category=fish | score=0.2966


Day 1 — Final Review

| Query                     | Top result  | Key observation                                      |
| ------------------------- | ----------- | ---------------------------------------------------- |
| Commonly kept as pets     | **Dog**     | Dog, cat and gecko all appear in the top 3           |
| Good swimmers             | **Penguin** | Bird and fish categories are retrieved together      |
| Predators                 | **Shark**   | Fish, mammal, bird and reptile all appear            |
| Social and live in groups | **Ant**     | Insect is retrieved based on social/colony semantics |


                 ┌──────────────────────┐
                 │      User Query      │
                 └──────────┬───────────┘
                            │
                            ▼
                 ┌──────────────────────┐
                 │ OpenAI Embedding     │
                 │ text-embedding-3-   │
                 │ small                │
                 │ 1536 dimensions     │
                 └──────────┬───────────┘
                            │
                            ▼
                 ┌──────────────────────┐
                 │       Qdrant         │
                 │ w7_animals_gorthi    │
                 │                      │
                 │ 10 vectors           │
                 │ + metadata/payload   │
                 └──────────┬───────────┘
                            │
                       Cosine similarity
                            │
                            ▼
                 ┌──────────────────────┐
                 │       Top-K          │
                 │ Relevant documents   │
                 │ + scores + metadata  │
                 └──────────────────────┘